In [1]:
"""
The code below generates dummy portfolios for testing. We can adjust the following parameters for each portfolio:
(1) The proportion of the portfolio that consists of one specific asset type (In the following, we had 50% UK CRE)
(2) The EAD, RSQ and LGD of all loans in the portfolio (we can not change these figures for one specific loan - do this in excel)
(3) The number of loans in the portfolio
The output of the following code are CSV files including the loan book, along with loan level CVaRs under a BINARY model. For each RSQ and proportion, we also generate a loss distribution figure.
Use this code if you need to automate the generation of dummy portfolios for varying RSQ values.
"""


import os
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from tqdm import tqdm

# Corporate color scheme variables
everyday_green = '#11B67A'
racing_green = '#024731'

def generate_loan_book(rsq_val, uk_cre_pct, total_loans=50000):
    n_uk_cre = int(total_loans * (uk_cre_pct / 50.0))
    n_other = total_loans - n_uk_cre
    other_categories = [
        ('UK', 'FI'), ('UK', 'Corporate'), 
        ('US', 'CRE'), ('US', 'FI'), ('US', 'Corporate')
    ]
    
    df_uk_cre = pd.DataFrame({
        'region': 'UK',
        'sector': 'CRE',
        'ead': 267220.0,
        'lgd': 0.3,
        'rsq': rsq_val,
        'credit_rating': 'BBB'
    }, index=range(n_uk_cre))
    
    choices = np.random.choice(len(other_categories), n_other)
    other_data = [other_categories[i] for i in choices]
    df_other = pd.DataFrame({
        'region': [x[0] for x in other_data],
        'sector': [x[1] for x in other_data],
        'ead': 267220.0,
        'lgd': 0.3,
        'rsq': rsq_val,
        'credit_rating': 'BBB'
    }, index=range(n_other))
    
    portfolio = pd.concat([df_uk_cre, df_other]).sample(frac=1).reset_index(drop=True)
    portfolio['loan_id'] = range(1, len(portfolio) + 1)
    
    column_order = ['loan_id', 'ead', 'lgd', 'credit_rating', 'rsq', 'sector', 'region']
    portfolio = portfolio[column_order]
    
    return portfolio

# ==========================================
# 2. Binary CVaR Simulator Class
# ==========================================
class BinaryCVaRSimulator:
    def __init__(self):
        self.portfolio_df = None
        self.cov = None
        self.C = None
        self.losses = None          
        self.all_loan_losses = None 
        self._analytic_el_val = 0.0

    def _make_pd(self, cov, eps=1e-8):
        cov = (cov + cov.T) / 2
        lam_min = np.linalg.eigvalsh(cov).min()
        delta = max(0.0, -lam_min + eps)
        return cov + delta * np.eye(cov.shape[0])
    
    def _chol_from_cov(self, cov, eps=1e-8):
        cov_pd = self._make_pd(cov, eps=eps)
        C = np.linalg.cholesky(cov_pd)
        return cov_pd, C

    def _build_factor_loadings(self, factor_index, M, N, C):
        F = np.zeros((M, N), dtype=float)
        for i, (_, row) in enumerate(self.portfolio_df[['region', 'sector']].iterrows()):
            F[i, factor_index[row['region']]] = 1.0
            F[i, factor_index[row['sector']]] = 1.0
        FC = F @ C
        row_norms = np.linalg.norm(FC, axis=1, keepdims=True)
        return FC / row_norms

    def simulate_losses(self, loan_book, pd_mapping, factor_index, cov, n_scenarios=100000, chunk=1000, seed=42):
        if 'loan_id' in loan_book.iloc[0].values:
            loan_book = loan_book.copy()
            loan_book.columns = loan_book.iloc[0]
            loan_book = loan_book.drop(loan_book.index[0]).reset_index(drop=True)
            
        for col in ['ead', 'lgd', 'rsq']:
            if col in loan_book.columns:
                loan_book[col] = pd.to_numeric(loan_book[col], errors='coerce')

        if 'pd' not in loan_book.columns:
            left_col = 'credit_rating'
            right_col = 'Credit Rating' if 'Credit Rating' in pd_mapping.columns else 'credit_rating'
            loan_book = pd.merge(loan_book, pd_mapping, left_on=left_col, right_on=right_col, how='left')
            if 'PD' in loan_book.columns and 'pd' not in loan_book.columns:
                loan_book.rename(columns={'PD': 'pd'}, inplace=True)
                
        self.portfolio_df = loan_book.reset_index(drop=True)
        self.cov, self.C = self._chol_from_cov(np.asarray(cov, float))
        
        M = len(self.portfolio_df)
        N = len(factor_index)

        ead = self.portfolio_df['ead'].to_numpy()
        lgd = self.portfolio_df['lgd'].to_numpy()
        pd_array = self.portfolio_df['pd'].to_numpy()
        rsq = self.portfolio_df['rsq'].to_numpy()
        
        weight = (ead * lgd)[:, None]
        thr = norm.ppf(np.clip(pd_array, 1e-12, 1.0 - 1e-12))[:, None]
        self._analytic_el_val = float(np.sum(pd_array * ead * lgd))

        sqrt_rsq = np.sqrt(np.clip(rsq, 0.0, 1.0))[:, None]
        FC_hat = self._build_factor_loadings(factor_index, M, N, self.C)

        rng = np.random.default_rng(seed)
        losses_out = np.empty(n_scenarios)
        filled = 0

        with tqdm(total=n_scenarios, desc='Pass 1/2: Simulating Binary Defaults') as pbar:
            while filled < n_scenarios:
                s = min(chunk, n_scenarios - filled)
                sl = slice(filled, filled + s)

                Z = rng.standard_normal(size=(N, s))
                Y = FC_hat @ Z
                Xi = rng.standard_normal(size=(M, s))
                X = sqrt_rsq * Y + np.sqrt(1.0 - sqrt_rsq ** 2) * Xi 

                defaults = (X <= thr)
                chunk_losses = defaults * weight
                losses_out[sl] = chunk_losses.sum(axis=0)
                
                filled += s
                pbar.update(s)

        self.losses = pd.Series(losses_out)
        
        var_threshold = float(np.quantile(losses_out, 0.99))
        tail_indices = np.where(losses_out >= var_threshold)[0]
        self.all_loan_losses = np.empty((M, len(tail_indices)))
        
        rng = np.random.default_rng(seed)
        filled = 0
        tail_saved_count = 0

        with tqdm(total=n_scenarios, desc='Pass 2/2: Mapping Loan Tail Risks') as pbar:
            while filled < n_scenarios:
                s = min(chunk, n_scenarios - filled)
                
                Z = rng.standard_normal(size=(N, s))
                Y = FC_hat @ Z
                Xi = rng.standard_normal(size=(M, s))
                X = sqrt_rsq * Y + np.sqrt(1.0 - sqrt_rsq ** 2) * Xi 

                defaults = (X <= thr)
                chunk_losses = defaults * weight

                current_tail_scenarios = tail_indices[(tail_indices >= filled) & (tail_indices < filled + s)]
                local_tail_indices = current_tail_scenarios - filled

                if len(local_tail_indices) > 0:
                    num_found = len(local_tail_indices)
                    self.all_loan_losses[:, tail_saved_count : tail_saved_count + num_found] = chunk_losses[:, local_tail_indices]
                    tail_saved_count += num_found

                filled += s
                pbar.update(s)

        return self.losses

    def analytic_EL(self):
        return self._analytic_el_val

    def var_quantile(self, alpha=0.99):
        return float(np.quantile(self.losses.to_numpy(), alpha))

    def expected_shortfall(self, alpha=0.99):
        arr = self.losses.to_numpy()
        q = np.quantile(arr, alpha)
        tail = arr[arr >= q]
        return float(tail.mean()) if len(tail) > 0 else float(q)

    def plot_loss_distribution(self, title_suffix=""):
        arr = self.losses.to_numpy()
        scale = 1_000_000.0

        fig, ax = plt.subplots(figsize=(6.75, 3.75), dpi=300)
        ax.hist(arr / scale, bins=60, color=everyday_green, edgecolor=None, alpha=0.85)

        ax.axvline(self.analytic_EL() / scale, color=racing_green, lw=2, label=f'Analytic EL (£{self.analytic_EL()/scale:,.2f}M)')
        ax.axvline(arr.mean() / scale, color=racing_green, lw=2, ls=':', label=f'Simulated EL (£{arr.mean()/scale:,.2f}M)')
        ax.axvline(self.var_quantile(0.95) / scale, color=racing_green, lw=2, ls='--', label=f'VaR 95% (£{self.var_quantile(0.95)/scale:,.2f}M)')
        ax.axvline(self.var_quantile(0.99) / scale, color=racing_green, lw=2, ls='-.', label=f'VaR 99% (£{self.var_quantile(0.99)/scale:,.2f}M)')

        ax.set_title(f'Portfolio Loss Distribution {title_suffix}')
        ax.set_xlabel('Absolute Portfolio Loss (£M)')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.set_axisbelow(True)
        ax.xaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter('£{x:,.0f}'))
        fig.tight_layout()
        return fig

    # --- MODIFIED: Now just returns the raw array instead of exporting a CSV ---
    def get_loan_level_cvar(self):
        # Calculate the mean of the tail losses for each loan
        loan_cvars_raw = self.all_loan_losses.mean(axis=1)
        return loan_cvars_raw

# ==========================================
# 3. Execution Block (The Automation Loop)
# ==========================================

output_dir = 'Results'
os.makedirs(output_dir, exist_ok=True)

# Load workspace files
systematic_shocks = pd.read_csv('systematic_shocks.csv', index_col='Quarter')
cov = systematic_shocks.cov()
factors = list(cov.columns)
factor_index = {name: j for j, name in enumerate(factors)}

pd_mapping = pd.read_csv('pd_mapping.csv')

# Define test parameters
rsq_values = [0.0, 0.2, 0.4, 0.6, 0.8]
uk_cre_pcts = [50, 90]

# Run the 12 combinations
for pct in uk_cre_pcts:
    for rsq_val in rsq_values:
        print(f"\n{'='*50}")
        print(f"RUNNING TEST: RSQ = {rsq_val} | UK CRE = {pct}%")
        print(f"{'='*50}")
        
        # Define filenames
        merged_filename = os.path.join(output_dir, f'Portfolio_{rsq_val}rsq{pct}.csv')
        fig_filename = os.path.join(output_dir, f'LossDist_{rsq_val}rsq{pct}.png')
        
        # 1. Generate Portfolio (Hold in memory, do not save yet)
        portfolio_df = generate_loan_book(rsq_val=rsq_val, uk_cre_pct=pct, total_loans=50000)
        
        # 2. Execute Simulation
        sim = BinaryCVaRSimulator()
        sim.simulate_losses(
            loan_book=portfolio_df,
            pd_mapping=pd_mapping,
            factor_index=factor_index,
            cov=cov,
            n_scenarios=10000, 
            chunk=1000 
        )
        
        # 3. Extract CVaR and Merge into Portfolio
        loan_cvars = sim.get_loan_level_cvar()
        portfolio_df['loan_level_CVaR'] = loan_cvars
        
        # 4. Save the fully merged CSV
        portfolio_df.to_csv(merged_filename, index=False)
        print(f"[*] Generated Merged File: {merged_filename}")
        
        # 5. Generate & Save Figure
        title_suffix = f"(RSQ={rsq_val}, UK CRE={pct}%)"
        fig = sim.plot_loss_distribution(title_suffix=title_suffix)
        fig.savefig(fig_filename, bbox_inches='tight', dpi=300)
        plt.close(fig) 
        print(f"[*] Saved Figure: {fig_filename}")

print("\nAll 12 tests completed successfully! Check the 'Results' folder for 12 CSVs and 12 PNGs.")


RUNNING TEST: RSQ = 0.0 | UK CRE = 50%


Pass 2/2: Mapping Loan Tail Risks: 100%|██████████| 10000/10000 [00:17<00:00, 583.83it/s]


[*] Generated Merged File: Results/Portfolio_0.0rsq50.csv
[*] Saved Figure: Results/LossDist_0.0rsq50.png

RUNNING TEST: RSQ = 0.2 | UK CRE = 50%


Pass 2/2: Mapping Loan Tail Risks: 100%|██████████| 10000/10000 [00:17<00:00, 574.25it/s]


[*] Generated Merged File: Results/Portfolio_0.2rsq50.csv
[*] Saved Figure: Results/LossDist_0.2rsq50.png

RUNNING TEST: RSQ = 0.4 | UK CRE = 50%


...

[*] Generated Merged File: Results/Portfolio_0.4rsq90.csv
[*] Saved Figure: Results/LossDist_0.4rsq90.png

RUNNING TEST: RSQ = 0.6 | UK CRE = 90%


Pass 2/2: Mapping Loan Tail Risks: 100%|██████████| 10000/10000 [00:17<00:00, 579.32it/s]


[*] Generated Merged File: Results/Portfolio_0.6rsq90.csv
[*] Saved Figure: Results/LossDist_0.6rsq90.png

RUNNING TEST: RSQ = 0.8 | UK CRE = 90%


Pass 2/2: Mapping Loan Tail Risks: 100%|██████████| 10000/10000 [00:17<00:00, 574.40it/s]


[*] Generated Merged File: Results/Portfolio_0.8rsq90.csv
[*] Saved Figure: Results/LossDist_0.8rsq90.png

All 12 tests completed successfully! Check the 'Results' folder for 12 CSVs and 12 PNGs.
